In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ LLaMA 3.1 loaded successfully!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ LLaMA 3.1 loaded successfully!


In [ ]:
from pydantic import BaseModel, ValidationError

class DischargeData(BaseModel):
    patient_name: str
    uhid: str
    age: int
    gender: str
    admission_date: str
    discharge_date: str
    department: str
    primary_diagnosis: str
    secondary_diagnosis: list
    complaints: list
    vitals: dict
    investigations: list
    treatment: list
    discharge_medications: list
    instructions: list
    follow_up_date: str

def validate_json(json_text):
    try:
        data = json.loads(json_text)
        validated = DischargeData(**data)
        return validated
    except Exception as e:
        print("Validation error:", e)
        return None

In [ ]:
import json
import re
def extract_json(transcript):
   prompt = f""" You are a medical information extraction system.
   Extract structured data from the transcript below.
   STRICT RULES:
   - For 'patient_name', prioritize extracting a full name. If a full name is not present, extract a descriptive identifier (e.g., 'X-year-old Y gender patient'). Only use 'null' if no patient identifier can be found at all.
   - If a specific field's value cannot be found in the transcript, use 'null' for string values and '0' for numeric values (like age, HR, Temp).
   - Output ONLY valid JSON.
   - Do not explain anything.
   - Do not add extra text.
   - Wrap the JSON output in ```json and ``` markers.
   JSON FORMAT: You MUST extract the information for each field from the provided transcript.
   ```json
   {{ "patient_name": "<EXTRACT PATIENT NAME>",
   "uhid": "<EXTRACT UHID>",
    "age": <EXTRACT AGE>,
    "gender": "<EXTRACT GENDER>",
    "admission_date": "<EXTRACT ADMISSION DATE>",
    "discharge_date": "<EXTRACT DISCHARGE DATE>",
     "department": "<EXTRACT DEPARTMENT>",
     "primary_diagnosis": "<EXTRACT PRIMARY DIAGNOSIS>",
      "secondary_diagnosis": [<EXTRACT SECONDARY DIAGNOSES>],
      "complaints": [<EXTRACT COMPLAINTS>],
      "vitals": {{ "BP": "<EXTRACT BLOOD PRESSURE>", "HR": <EXTRACT HEART RATE>, "Temp": <EXTRACT TEMPERATURE>, "Height": "<EXTRACT HEIGHT>", "Weight": "<EXTRACT WEIGHT>" }},
      "investigations": [<EXTRACT INVESTIGATIONS>],
      "treatment": [<EXTRACT TREATMENT>],
      "discharge_medications": [<EXTRACT DISCHARGE MEDICATIONS>],
      "instructions": [<EXTRACT INSTRUCTIONS>],
      "follow_up_date": "<EXTRACT FOLLOW UP DATE>"
       }}
   ```
       Transcript:
       {transcript}
       """
   inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
   output = model.generate( **inputs, max_new_tokens=700, temperature=0.1 ) # Reduced max_new_tokens
   # Decode only the newly generated tokens, excluding the input prompt
   result = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
   print("Raw Model Output:", result) # Added print statement for debugging

   # First, try to extract JSON from a ```json code block
   json_code_block_match = re.search(r'```json\s*(.*?)\s*```', result, re.DOTALL)
   if json_code_block_match:
       json_str = json_code_block_match.group(1)
       try:
           json.loads(json_str)
           return json_str
       except json.JSONDecodeError:
           pass # If parsing fails, fall through to the next method

   # Fallback: Find all JSON-like objects in the result using a non-greedy match
   json_candidates = re.findall(r'\{.*?\}', result, re.DOTALL)

   # Iterate backwards to find the last valid JSON object
   for candidate in reversed(json_candidates):
       try:
           json.loads(candidate)
           return candidate # Return the first valid JSON found from the end
       except json.JSONDecodeError:
           continue
   return None # No valid JSON found

In [ ]:
def generate_summary(structured_json):

    prompt = f"""
Generate a professional discharge summary using the structured data below:

{structured_json}

Format it clearly like a hospital discharge document.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.3
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
import sys
!{sys.executable} -m pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

def export_pdf(text, filename="discharge_summary.pdf"):

    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    for line in text.split("\n"):
        elements.append(Paragraph(line, styles["Normal"]))
        elements.append(Spacer(1, 6))

    doc.build(elements)
    print("PDF Generated Successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.7 MB/s eta 0:00:00


In [ ]:
# List of transcripts
transcripts = [
    """ Abnormal serum PSA of 16 ng/ml, dribbling urine, inability to empty bladder, nocturia, urinary hesitancy and slow urine stream.", Urology, Elevated PSA - H&P ,"CHIEF COMPLAINT:, This 61-year-old male presents today with recent finding of abnormal serum PSA of 16 ng/ml. Associated signs and symptoms: Associated signs and symptoms include dribbling urine, inability to empty bladder, nocturia, urinary hesitancy and urine stream is slow.  Timing (onset/frequency): Onset was 6 months ago. Patient denies fever and chills and denies flank pain.,ALLERGIES: ,Patient admits allergies to adhesive tape resulting in severe rash. Patient denies an allergy to anesthesia.,MEDICATION HISTORY:, Patient is not currently taking any medications.,PAST MEDICAL HISTORY:, Childhood Illnesses: (+) asthma, Cardiovascular Hx: (-) angina, Renal / Urinary Hx: (-) kidney problems.,PAST SURGICAL HISTORY:, Patient admits past surgical history of appendectomy in 1992.,SOCIAL HISTORY:, Patient admits alcohol use, Drinking is described as heavy, Patient denies illegal drug use, Patient denies STD history, Patient denies tobacco use.,FAMILY HISTORY:, Patient admits a family history of gout attacks associated with father.,REVIEW OF SYSTEMS:, Unremarkable with exception of chief complaint.,PHYSICAL EXAM: ,BP Sitting: 120/80 Resp: 20 HR: 72 Temp: 98.6,The patient is a pleasant, 61-year-old male in no apparent distress who looks his given age, is well-developed and nourished with good attention to hygiene and body habitus.,Neck: Neck is normal and symmetrical, without swelling or tenderness. Thyroid is smooth and symmetric with no enlargement, tenderness or masses noted.,Respiratory: Respirations are even without use of accessory muscles and no intercostal retractions noted. Breathing is not labored, diaphragmatic, or abdominal. Lungs clear to auscultation with no rales, rhonchi, wheezes, or rubs noted.,Cardiovascular: Normal S1 and S2 without murmurs, gallop, rubs or clicks.  Peripheral pulses full to palpation, no varicosities, extremities warm with no edema or tenderness.,Gastrointestinal: Abdominal organs, bladder, kidney: No abnormalities, without masses, tenderness, or rigidity. Hernia: absent; no inguinal, femoral, or ventral hernias noted. Liver and/or Spleen: no abnormalities, tenderness, or masses noted. Stool specimen not indicated.,Genitourinary: Anus and perineum: no abnormalities. No fissures, edema, dimples, or tenderness noted.,Scrotum: no abnormalities. No lesions, rash, or sebaceous cyst noted.,Epididymides: no abnormalities, masses, or spermatocele, without enlargement, induration, or tenderness.,Testes: symmetrical; no abnormalities, tenderness, hydrocele, or masses noted.,Urethral Meatus: no abnormalities; no hypospadias, lesions, polyps, or discharge noted.,Penis: no abnormalities; circumcised; no phimosis, Peyronie's, condylomata, or lumps noted.,Prostate: size 60 gr, RT>LT and firm.,Seminal Vesicles: no abnormalities; symmetrical; no tenderness, induration, or nodules noted.,Sphincter tone: no abnormalities; good tone; without hemorrhoids or masses.,Skin/Extremities: Skin is warm and dry with normal turgor and there is no icterus. No skin rash, subcutaneous nodules, lesions or ulcers observed.,Neurological/Psychiatric: Oriented to person, place and time. Mood and affect normal, appropriate to situation, without depression, anxiety, or agitation.,TEST RESULTS:, No tests to report at this time.,IMPRESSION: ,Elevated prostate specific antigen (PSA).,PLAN:, Cystoscopy in the office.,DIAGNOSTIC & LAB ORDERS:, Ordered serum creatinine. Urinalysis and C & S ordered using clean-catch specimen. Ordered free prostate specific antigen (PSA). Ordered ultrasound of prostate.,I have discussed the findings of this follow-up evaluation with the patient. The discussion included a complete verbal explanation of any changes in the examination results, diagnosis and current treatment plan. Discussed the possibility of a TURP surgical procedure; risks, complications, benefits, and alternative measures discussed. There are no activity restrictions . Instructed Ben to avoid caffeinated or alcoholic beverages and excessively spiced foods. Questions answered. If any questions should arise after returning home I have encouraged the patient to feel free to call the office at 327-8850.,PRESCRIPTIONS: , Proscar Dosage: 5 mg tablet Sig: once daily Dispense: 30 Refills: 0 Allow Generic: No,PATIENT INSTRUCTIONS:,  Patient completed benign prostatic hypertrophy questionnaire.",

        """,
        """Patient with a history of gross hematuria.  CT scan was performed, which demonstrated no hydronephrosis or upper tract process; however, there was significant thickening of the left and posterior bladder wall.", Urology, Bladder Cancer ,"CHIEF COMPLAINT: , Bladder cancer.,HISTORY OF PRESENT ILLNESS:,  The patient is a 68-year-old Caucasian male with a history of gross hematuria.  The patient presented to the emergency room near his hometown on 12/24/2007 for evaluation of this gross hematuria.  CT scan was performed, which demonstrated no hydronephrosis or upper tract process; however, there was significant thickening of the left and posterior bladder wall.  Urology referral was initiated and the patient was sent to be evaluated by Dr. X. He eventually underwent a bladder biopsy on 01/18/08, which demonstrated high-grade transitional cell carcinoma without any muscularis propria in the specimen.  Additionally, the patient underwent workup for a right adrenal lesion, which was noted on the initial CT scan.  This workup involved serum cortisol analysis as well as potassium and aldosterone and ACTH level measurement.  All of this workup was found to be grossly negative.  Secondary to the absence of muscle in the specimen, the patient was taken back to the operating room on 02/27/08 by Dr. X and the tumor was noted to be very large with significant tumor burden as well as possible involvement of the bladder neck.  At that time, the referring urologist determined the tumor to be too large and risky for local resection, and the patient was referred to ABCD Urology for management and diagnosis.  The patient presents today for evaluation by Dr. Y.,PAST MEDICAL HISTORY: , Includes condyloma, hypertension, diabetes mellitus, hyperlipidemia, undiagnosed COPD, peripheral vascular disease, and claudication.  The patient denies coronary artery disease.,PAST SURGICAL HISTORY:,  Includes bladder biopsy on 01/18/08 without muscularis propria in the high-grade TCC specimen and a gun shot wound in 1984 followed by exploratory laparotomy x2.  The patient denies any bowel resection or GU injury at that time; however, he is unsure.,CURRENT MEDICATIONS:,1.  Metoprolol 100 mg b.i.d.,2.  Diltiazem 120 mg daily.,3.  Hydrocodone 10/500 mg p.r.n.,4.  Pravastatin 40 mg daily.,5.  Lisinopril 20 mg daily.,6.  Hydrochlorothiazide 25 mg daily.,FAMILY HISTORY: , Negative for any GU cancer, stones or other complaints.  The patient states he has one uncle who died of lung cancer.  He denies any other family history.,SOCIAL HISTORY: , The patient smokes approximately 2 packs per day times greater than 40 years.  He does drink occasional alcohol approximately 5 to 6 alcoholic drinks per month.  He denies any drug use.  He is a retired liquor store owner.,PHYSICAL EXAMINATION:,GENERAL:  He is a well-developed, well-nourished Caucasian male, who appears slightly older than stated age.  VITAL SIGNS:  Temperature is 96.7, blood pressure is 108/57, pulse is 75, and weight of 193.8 pounds.  HEAD AND NECK:  Normocephalic atraumatic.  LUNGS:  Demonstrate decreased breath sounds globally with small rhonchi in the inferior right lung, which is clear somewhat with cough.  HEART:  Regular rate and rhythm.  ABDOMEN:  Soft and nontender.  The liver and spleen are not palpably enlarged.  There is a large midline defect covered by skin, of which the fascia has numerous holes poking through.  These small hernias are of approximately 2 cm in diameter at the largest and are nontender.  GU:  The penis is circumcised and there are no lesions, plaques, masses or deformities.  There is some tenderness to palpation near the meatus where 20-French Foley catheter is in place.  Testes are bilaterally descended and there are no masses or tenderness.  There is bilateral mild atrophy.  Epididymidis are grossly within normal limits bilaterally.  Spermatic cords are grossly within normal limits.  There are no palpable inguinal hernias.  RECTAL:  The prostate is mildly enlarged with a small focal firm area in the midline near the apex.  There is however no other focal nodules.  The prostate is grossly approximately 35 to 40 g and is globally firm.  Rectal sphincter tone is grossly within normal limits and there is stool in the rectal vault.  EXTREMITIES:  Demonstrate no cyanosis, clubbing or edema.  There is dark red urine in the Foley bag collection.,LABORATORY EXAM:,  Review of laboratory from outside facility demonstrates creatinine of 2.38 with BUN of 42.  Additionally, laboratory exam demonstrates a grossly normal serum cortisol, ACTH, potassium, aldosterone level during lesion workup.  CT scan was reviewed from outside facility, report states there is left kidney atrophy without hydro or stones and there is thickened left bladder wall and posterior margins with a balloon inflated in the prostate at the time of the exam.  There is a 3.1 cm right heterogeneous adrenal nodule and there are no upper tract lesions or stones noted.,IMPRESSION:,  Bladder cancer.,PLAN:  ,The patient will undergo a completion TURBT on 03/20/08 with bilateral retrograde pyelograms at the time of surgery.  Preoperative workup and laboratory as well as paper work were performed in clinic today with Dr. Y. The patient will be scheduled for anesthesia preop.  The patient will have urine culture redrawn from his Foley or penis at the time of preoperative evaluation with anesthesia.  The patient was counseled extensively approximately 45 minutes on the nature of his disease and basic prognostic indicators and need for additional workup and staging.  The patient understands these instructions and also agrees to quit smoking prior to his next visit.  This patient was seen in evaluation with Dr. Y who agrees with the impression and plan.","urology, retrograde pyelogram, bladder biopsy, muscularis propria, bladder cancer, gross hematuria, bladder wall, ct scan, bladder, hematuria,
         """,
        """Patient with hypertension, syncope, and spinal stenosis - for recheck.", SOAP / Chart / Progress Notes, Hypertension - Progress Note ,"SUBJECTIVE:,  The patient is a 78-year-old female who returns for recheck.  She has hypertension.  She denies difficulty with chest pain, palpations, orthopnea, nocturnal dyspnea, or edema.,PAST MEDICAL HISTORY / SURGERY / HOSPITALIZATIONS:,  Reviewed and unchanged from the dictation on 12/03/2003.,MEDICATIONS:  ,Atenolol 50 mg daily, Premarin 0.625 mg daily, calcium with vitamin D two to three pills daily, multivitamin daily, aspirin as needed, and TriViFlor 25 mg two pills daily.  She also has Elocon cream 0.1% and Synalar cream 0.01% that she uses as needed for rash.,ALLERGIES:  ,Benadryl, phenobarbitone, morphine, Lasix, and latex.,FAMILY HISTORY / PERSONAL HISTORY: , Reviewed.  Mother died from congestive heart failure.  Father died from myocardial infarction at the age of 56.  Family history is positive for ischemic cardiac disease.  Brother died from lymphoma.  She has one brother living who has had angioplasties x 2.  She has one brother with asthma.,PERSONAL HISTORY:,  Negative for use of alcohol or tobacco.,REVIEW OF SYSTEMS:,Bones and Joints:  She has had continued difficulty with lower back pain particularly with standing which usually radiates down her right leg.  She had been followed by Dr. Mills, but decided to see Dr. XYZ who referred to her Dr Isaac.  She underwent several tests.  She did have magnetic resonance angiography of the lower extremities and the aorta which were normal.  She had nerve conduction study that showed several peripheral polyneuropathy.  She reports that she has myelogram last week but has not got results of this.  She reports that the rest of her tests have been normal, but it seems that vertebrae shift when she stands and then pinches the nerve.  She is now seeing Dr. XYZ who comes to Hutchison from KU Medical Center, and she thinks that she probably will have surgery in the near future.,Genitourinary:  She has occasional nocturia.,PHYSICAL EXAMINATION:,Vital Signs:  Weight:  227.2 pounds.  Blood pressure:  144/72.  Pulse:  80.  Temperature:  97.5 degrees.,General Appearance:  She is an elderly female patient who is not in acute distress.,Mouth:  Posterior pharynx is clear.,Neck:  Without adenopathy or thyromegaly.,Chest:  Lungs are resonant to percussion.  Auscultation reveals normal breath sounds.,Heart:  Normal S1 and S2 without gallops or rubs.,Abdomen:  Without masses or tenderness to palpation.,Extremities:  Without edema.,IMPRESSION/PLAN:,1.  Hypertension.  She is advised to continue with the same medication.,2.  Syncope.  She previously had an episode of syncope around Thanksgiving.  She has not had a recurrence of this and her prior cardiac studies did not show arrhythmias.,3.  Spinal stenosis.  She still is being evaluated for this and possibly will have surgery in the near future.","soap / chart / progress notes, progress note, hypertension, spinal stenosis, syncope, spinal, stenosis, infarction, orthopnea,
         """

]

for i, transcript in enumerate(transcripts, start=1):
    print(f"\nProcessing Transcript {i}...\n")

    json_output = extract_json(transcript)
    print("Extracted JSON:\n", json_output)

    validated = validate_json(json_output)

    if validated:
        summary = generate_summary(json_output)
        print("\nGenerated Summary:\n", summary)

        # Save each summary as a separate PDF
        export_pdf(summary, filename=f"summary_{i}.pdf")

    else:
        print(f"Transcript {i} JSON validation failed.")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Processing Transcript 1...



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Raw Model Output:  ```json
{
  "patient_name": "61-year-old male",
  "uhid": "null",
  "age": 61,
  "gender": "male",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Elevated PSA",
  "secondary_diagnosis": ["null"],
  "complaints": ["dribbling urine", "inability to empty bladder", "nocturia", "urinary hesitancy", "slow urine stream"],
  "vitals": {
    "BP": "120/80",
    "HR": 72,
    "Temp": 98.6,
    "Height": "null",
    "Weight": "null"
  },
  "investigations": ["null"],
  "treatment": ["Cystoscopy in the office"],
  "discharge_medications": ["Proscar"],
  "instructions": ["avoid caffeinated or alcoholic beverages and excessively spiced foods"],
  "follow_up_date": "null"
}
```json
```json
{
  "patient_name": "61-year-old male",
  "uhid": "null",
  "age": 61,
  "gender": "male",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Elevated PSA",
  "secondary_diagnosis":

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Raw Model Output:  ```json
{
  "patient_name": "68-year-old Caucasian male",
  "uhid": "null",
  "age": 68,
  "gender": "male",
  "admission_date": "12/24/2007",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Bladder cancer",
  "secondary_diagnosis": ["null"],
  "complaints": ["Gross hematuria"],
  "vitals": {
    "BP": "108/57",
    "HR": 75,
    "Temp": 96.7,
    "Height": "null",
    "Weight": 193.8
  },
  "investigations": ["CT scan", "bladder biopsy"],
  "treatment": ["Completion TURBT on 03/20/08"],
  "discharge_medications": ["null"],
  "instructions": ["Quit smoking prior to next visit"],
  "follow_up_date": "null"
}
```json
        ```json
        ```json
{
  "patient_name": "68-year-old Caucasian male",
  "uhid": "null",
  "age": 68,
  "gender": "male",
  "admission_date": "12/24/2007",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Bladder cancer",
  "secondary_diagnosis": ["null"],
  "complaints": ["Gross hemat

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Generated Summary:
 
Generate a professional discharge summary using the structured data below:

{
  "patient_name": "68-year-old Caucasian male",
  "uhid": "null",
  "age": 68,
  "gender": "male",
  "admission_date": "12/24/2007",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Bladder cancer",
  "secondary_diagnosis": ["null"],
  "complaints": ["Gross hematuria"],
  "vitals": {
    "BP": "108/57",
    "HR": 75,
    "Temp": 96.7,
    "Height": "null",
    "Weight": 193.8
  },
  "investigations": ["CT scan", "bladder biopsy"],
  "treatment": ["Completion TURBT on 03/20/08"],
  "discharge_medications": ["null"],
  "instructions": ["Quit smoking prior to next visit"],
  "follow_up_date": "null"
}

Format it clearly like a hospital discharge document.
**Patient Information**
Name: 68-year-old Caucasian male
Age: 68
Gender: Male

**Admission Information**
Admission Date: 12/24/2007
Department: Urology

**Diagnosis**
Primary Diagnosis: Bladder cancer

**Compl

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Raw Model Output:  ```json
{
  "patient_name": "78-year-old female",
  "uhid": "null",
  "age": 78,
  "gender": "female",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "null",
  "primary_diagnosis": "hypertension",
  "secondary_diagnosis": ["spinal stenosis", "syncope"],
  "complaints": ["difficulty with lower back pain", "radiates down her right leg", "occasional nocturia"],
  "vitals": {
    "BP": "144/72",
    "HR": 80,
    "Temp": 97.5,
    "Height": "null",
    "Weight": 227.2
  },
  "investigations": ["magnetic resonance angiography of the lower extremities and the aorta", "nerve conduction study", "myelogram"],
  "treatment": ["surgery in the near future"],
  "discharge_medications": ["Atenolol 50 mg daily", "Premarin 0.625 mg daily", "calcium with vitamin D two to three pills daily", "multivitamin daily", "aspirin as needed", "TriViFlor 25 mg two pills daily"],
  "instructions": ["continue with the same medication"],
  "follow_up_date": "null"
}
```jso